# Amaliyot 12. K-Means

12-interaktiv laboratoriyaga («K-Means algoritmi») hamroh. Kurs mavzusi: matnlarni avtomatik guruhlash va parametrlar taʼsiri

Daftar oʻz-oʻzidan yetarli: birinchi katak `common.py` faylini oʻquv korpusi va yordamchi funksiyalar bilan yaratadi. Kataklarni yuqoridan pastga ishga tushiring. Korpus oʻquv (shablon) boʻlib, haqiqiy masalada oʻz CSV faylingizni quyidagi ustunlar bilan qoʻying: `text` va `label`.

In [ ]:
%%writefile common.py
"""Matnlarni tasniflash va klasterlash amaliyotlari uchun umumiy funksiyalar."""
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

TOPICS = {
    "sport": ("oʻyin jamoa gol oʻyinchi murabbiy chempionat turnir gʻalaba magʻlubiyat mavsum stadion muxlis hakam hisob "
              "himoyachi hujumchi darvozabon kubok liga final transfer klub zarba penalti musobaqa sportchi "
              "rekord medal olimpiada mashq durang").split(),
    "siyosat": ("qonun deputat saylov partiya ovoz prezident vazir hukumat parlament islohot byudjet "
                "sanksiya nomzod koalitsiya vazirlik farmon tuzatish loyiha muxolifat referendum diplomatiya "
                "hokim konstitutsiya majlis komissiya muzokara sammit saylovchi mandat vakolat").split(),
    "oshxona": ("shoʻrva retsept tuz goʻsht pishirmoq duxovka sabzavot tova qozon xamir shakar ziravor qaynatmoq qovurmoq "
                "dimlamoq sous bulyon salat oshpaz taom masalliq yogʻ un tuxum sut sarimsoq piyoz garmdori "
                "kartoshka shirinlik pishiriq marinad").split(),
    "texnologiya": ("protsessor dastur ilova smartfon neyrotarmoq algoritm maʼlumot server yangilanish dasturchi "
                    "kompyuter xotira ekran kamera brauzer tarmoq internet bulut kod platforma foydalanuvchi qurilma "
                    "chip batareya noutbuk dasturiy videokarta tizim interfeys shifrlash").split(),
}
NEUTRAL = ("bugun juda yangi aytishadi keyin ham mumkin edi boʻldi maʼlum qilindi kecha yaqinda hali hammasi "
           "bir nechta muhim birinchi").split()


def make_corpus(n_per_class=150, noise=0.3, length=(6, 11), seed=0):
    """4 mavzudagi qisqa «hujjatlar»dan iborat oʻquv korpusi.
    noise: oʻz mavzusidan boʻlmagan soʻzlar ulushi (neytral yoki begona). noise qancha katta boʻlsa, masala shuncha qiyin."""
    rng = random.Random(seed)
    names = list(TOPICS)
    rows = []
    for label in names:
        others = [t for t in names if t != label]
        for _ in range(n_per_class):
            words = []
            for _ in range(rng.randint(*length)):
                u = rng.random()
                if u < noise * 0.5:
                    words.append(rng.choice(NEUTRAL))
                elif u < noise:
                    words.append(rng.choice(TOPICS[rng.choice(others)]))
                else:
                    words.append(rng.choice(TOPICS[label]))
            rows.append((" ".join(words), label))
    df = pd.DataFrame(rows, columns=["text", "label"])
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)


def project_2d(X, seed=0):
    """TF-IDF siyrak matritsasini tekislikka proyeksiyalash (TruncatedSVD)."""
    from sklearn.decomposition import TruncatedSVD
    return TruncatedSVD(2, random_state=seed).fit_transform(X)


def scatter2d(Z, labels, title="", ax=None):
    ax = ax or plt.gca()
    codes, uniq = pd.factorize(pd.Series(labels))
    sc = ax.scatter(Z[:, 0], Z[:, 1], c=codes, cmap="tab10", s=14, alpha=0.8)
    ax.set_title(title)
    handles = [plt.Line2D([], [], marker="o", ls="", color=sc.cmap(sc.norm(i)), label=str(u)) for i, u in enumerate(uniq)]
    ax.legend(handles=handles, fontsize=8)
    return ax


def top_terms(coef, terms, class_names, n=8):
    """Har bir sinf uchun eng katta vaznli soʻzlarni chiqaradi (chiziqli modellar)."""
    terms = np.asarray(terms)
    for i, name in enumerate(class_names):
        idx = np.argsort(coef[i])[::-1][:n]
        print(f"{name:12s}", ", ".join(terms[idx]))


In [ ]:
from common import *

# Oʻquv korpusi: 4 ta mavzu. noise — "begona" soʻzlar ulushi (qancha koʻp boʻlsa, shuncha qiyin).
df = make_corpus(n_per_class=150, noise=0.45, seed=0)
# Oʻquv korpusi oʻrniga oʻzingiznikini qoʻying: df = pd.read_csv("texts.csv")  # ustunlar: text, label
print(df.label.value_counts().to_dict())
df.head()

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
vec = TfidfVectorizer(); X = vec.fit_transform(df.text); y = df.label.values
terms = np.array(vec.get_feature_names_out())

## 1. Klasterlar va ularning kalit soʻzlari

In [ ]:
km = KMeans(4, init="k-means++", n_init=10, random_state=0).fit(X)
print("ARI:", round(adjusted_rand_score(y, km.labels_), 3), "| inertia:", round(km.inertia_, 2))
for c in range(4):
    print(f"Klaster {c}:", ", ".join(terms[km.cluster_centers_[c].argsort()[::-1][:6]]), "|", pd.Series(y[km.labels_ == c]).value_counts().to_dict())

## 2. Initsializatsiya: lokal minimumlar
Bitta ishga tushirish (`n_init=1`) turli seed lar bilan: tasodifiy va k-means++ initsializatsiya.

In [ ]:
res = {}
for init in ["random", "k-means++"]:
    res[init] = [KMeans(4, init=init, n_init=1, random_state=s).fit(X).inertia_ for s in range(30)]
pd.DataFrame(res).plot.box(); plt.ylabel("inertia (kamroq yaxshiroq)"); plt.title("Initsializatsiya sabab natija tarqoqligi"); plt.show()
print(pd.DataFrame(res).describe().round(2))

## 3. Tirsak usuli

In [ ]:
ks = range(1, 11)
inertia = [KMeans(k, n_init=5, random_state=0).fit(X).inertia_ for k in ks]
plt.plot(ks, inertia, marker="o"); plt.xlabel("k"); plt.ylabel("inertia"); plt.title("Tirsak usuli"); plt.show()

## 4. 2D da rasm

In [ ]:
from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(2, random_state=0); Z = svd.fit_transform(X); C = svd.transform(km.cluster_centers_)
plt.scatter(Z[:, 0], Z[:, 1], c=km.labels_, cmap="tab10", s=12, alpha=.7); plt.scatter(C[:, 0], C[:, 1], marker="X", s=200, c="black")
plt.title("K-Means: klasterlar va sentroidlar (SVD proyeksiyasi)"); plt.show()

## Topshiriqlar

1. ARI maksimal boʻladigan k ni toping. U «tirsak» bilan mos tushadimi?
2. Sinab koʻring `MiniBatchKMeans`. Natija qanchalik farq qiladi va necha marta tezroq?
3. Nima uchun matnlar uchun vektorlarni normallashtirish foydali (sferik K-Means)?